# I08 - Encoder-Decoder and Seq2Seq Models

**Author:** Karthik Arjun  
**LinkedIn:** [karthik-arjun-a5b4a258](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Created:** March 2026  
**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand encoder-decoder architecture
2. Implement sequence-to-sequence (seq2seq) models
3. Learn attention mechanisms for seq2seq
4. Build machine translation systems
5. Apply seq2seq to text summarization

---

## Prerequisites

- Completed I07 (Advanced RNN Architectures)
- Understanding of RNNs, LSTMs, GRUs
- Familiarity with sequence modeling

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. Encoder-Decoder Architecture

### Concept

**Problem:** Variable-length input → Variable-length output
- Translation: "Hello" → "Bonjour"
- Summarization: Long text → Short summary
- Question answering: Question → Answer

**Solution:** Encoder-Decoder (Seq2Seq)

### Architecture Components

**1. Encoder:**
- Processes input sequence
- Compresses into fixed-size context vector
- $h_t = f(h_{t-1}, x_t)$
- Final state = context vector $c$

**2. Context Vector:**
- Fixed-size representation of input
- Contains all information from input
- Bottleneck of the architecture

**3. Decoder:**
- Generates output sequence
- Initialized with context vector
- $s_t = g(s_{t-1}, y_{t-1}, c)$
- Outputs one token at a time

### Mathematical Formulation

**Encoder:**
$h_t = f_{enc}(h_{t-1}, x_t)$
$c = h_T$ (final hidden state)

**Decoder:**
$s_0 = c$ (initialize with context)
$s_t = f_{dec}(s_{t-1}, y_{t-1})$
$p(y_t | y_{<t}, x) = softmax(W_s s_t)$

In [ ]:
def create_seq2seq_model(input_vocab_size=10000, output_vocab_size=10000,
                        embedding_dim=256, latent_dim=512):
    """Create basic seq2seq model without attention
    
    Args:
        input_vocab_size: Source vocabulary size
        output_vocab_size: Target vocabulary size
        embedding_dim: Embedding dimension
        latent_dim: LSTM hidden dimension
    
    Returns:
        Training model, encoder model, decoder model
    """
    # Encoder
    encoder_inputs = keras.Input(shape=(None,), name='encoder_input')
    encoder_embedding = layers.Embedding(input_vocab_size, embedding_dim)(encoder_inputs)
    encoder_lstm = layers.LSTM(latent_dim, return_state=True, name='encoder_lstm')
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
    encoder_states = [state_h, state_c]
    
    # Decoder
    decoder_inputs = keras.Input(shape=(None,), name='decoder_input')
    decoder_embedding = layers.Embedding(output_vocab_size, embedding_dim)(decoder_inputs)
    decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True, 
                               name='decoder_lstm')
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
    decoder_dense = layers.Dense(output_vocab_size, activation='softmax', name='decoder_output')
    decoder_outputs = decoder_dense(decoder_outputs)
    
    # Training model
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs, name='Seq2Seq')
    
    # Encoder model for inference
    encoder_model = Model(encoder_inputs, encoder_states, name='Encoder')
    
    # Decoder model for inference
    decoder_state_input_h = keras.Input(shape=(latent_dim,))
    decoder_state_input_c = keras.Input(shape=(latent_dim,))
    decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
    
    decoder_embedding_inf = layers.Embedding(output_vocab_size, embedding_dim)
    decoder_inputs_single = keras.Input(shape=(1,))
    decoder_embedding_inf_out = decoder_embedding_inf(decoder_inputs_single)
    
    decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
        decoder_embedding_inf_out, initial_state=decoder_states_inputs)
    decoder_states_inf = [state_h_inf, state_c_inf]
    decoder_outputs_inf = decoder_dense(decoder_outputs_inf)
    
    decoder_model = Model(
        [decoder_inputs_single] + decoder_states_inputs,
        [decoder_outputs_inf] + decoder_states_inf,
        name='Decoder'
    )
    
    return model, encoder_model, decoder_model

seq2seq_model, encoder_model, decoder_model = create_seq2seq_model()
print("Seq2Seq model created")
print("\nTraining model: Takes encoder input + decoder input → decoder output")
print("Encoder model: For inference (encode input)")
print("Decoder model: For inference (decode step-by-step)")

seq2seq_model.summary()

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I08)
- [ ] Encoder-decoder architecture
- [ ] Attention in seq2seq
- [ ] Beam search
- [ ] Teacher forcing

---